In [1]:
from ovo import db, storage, schedulers, descriptors, design_logic, descriptor_logic, \
    models_proteinqc, project_logic, descriptors_proteinqc
from ovo.app.components import molstar_notebook, StructureVisualization
import os
import seaborn as sns
from matplotlib import pyplot as plt

%config InlineBackend.figure_format = 'retina'

Registering plugin ovo_promb
Registering plugin ovo_proteindj


OVO home /home/username/ovo

In [2]:
project, project_round = project_logic.get_or_create_project_round("OVO Publication Examples 1", "Motif scaffolding")

In [3]:
pools = db.Pool.select(round_id=project_round.id)
for pool in pools:
    print(pool.id, pool.name)

ogc 1A41 6*100*8 designs active site weights
jov 1A41 6*100*8 designs default weights


In [4]:
# Pool id from previous notebooks
POOL_IDS = [p.id for p in pools]

In [5]:
designs = db.Design.select(pool_id__in=POOL_IDS, accepted=True)
len(designs)

26

In [6]:
workflow = models_proteinqc.ProteinQCWorkflow(
    chains=['A'],
    design_ids=[design.id for design in designs],
    tools=['seq_composition', 'esm_1v', 'esm_if', 'dssp', 'peppatch', 'proteinsol'],
)
workflow.validate()
workflow.get_table_row()

Workflow  type    ProteinQC workflow
dtype: object

In [7]:
print(schedulers.keys())

SCHEDULER_KEY = 'slurm_singularity'

dict_keys(['slurm_singularity', 'pbs_singularity', 'local_singularity', 'local_conda', 'local_single_gpu'])


In [8]:
#
# SUBMIT - note that running this multiple times will submit a the workflow each time!
#
descriptor_job = descriptor_logic.submit_descriptor_workflow(
    workflow=workflow,
    scheduler_key=SCHEDULER_KEY,
    project_id=project.id
)
descriptor_job.id

Submitting workflow: nextflow run -with-trace trace.txt -work-dir /home/username/ovo/workdir/work /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc --publish_dir output --reference_files_dir /home/username/ovo/reference_files --shared_modules ovo:/home/username/projects/ovo-open-source/ovo/ovo,ovo_promb:/home/username/projects/ovo-open-source/ovo-promb/ovo_promb,ovo_proteindj:/home/username/projects/ovo-proteindj/ovo_proteindj -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/nextflow_default.config -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc/nextflow.config -profile singularity -config /home/username/ovo/nextflow_slurm_singularity.config -ansi-log false -bg --input_pdb /home/username/ovo/workdir/inputs/dc/c196025b2e7656d1153b3831b1c6e6ef3e2969/input_pdb_paths.txt --tools seq_composition,esm_1v,esm_if,dssp,peppatch,proteinsol --chains A --batch_size 50
Execution directory: /home/username/ovo/workdir/execdir/e164e85a-c168

'54a3b4'

In [15]:
descriptor_logic.process_results(descriptor_job)

Job already processed


In [10]:
db.DescriptorValue.count(design_id__in=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True))

3926

In [11]:
values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True),
)
print(len(values), 'designs')
print(len(values.columns), 'descriptors')
values.head()

26 designs
110 descriptors


,Sequence A,AF2 ipTM score,AF2 pTM score,AF2 Design RMSD,AF2 Native Motif RMSD,AF2 PAE,AF2 pLDDT,AlphaFold2 Initial Guess prediction,Radius of gyration (backbone),PyDSSP String,...,Normalized positive top1 patch area at pH 7.4,Negative patch area at pH 7.4,Normalized negative patch area at pH 7.4,Negative top1 patch area at pH 7.4,Normalized negative top1 patch area at pH 7.4,Electrostatic volume integral at pH 7.4,Positive electrostatic regions volume integral at pH 7.4,Negative electrostatic regions volume integral at pH 7.4,Positive electrostatic volume integral at pH 7.4,Negative electrostatic volume integral at pH 7.4
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_ogc_01_053_seq01,MITLILVADTSAASREKIVVITKALKEKLGVETIGHLLTLSPDSLE...,0.0,0.823697,1.158534,1.292619,3.961928,89.657974,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,13.938089,-EEEEEEE----HHHHHHHHHHHHHHHHH--EEEEEEE------HH...,...,0.015935,2144.994972,0.281221,1593.961902,0.208978,-43521.721970,2657.710732,-33099.978712,6058.603038,-49580.325008
ovo_ogc_01_067_seq04,MVDTVIATPLSEEAQVKLEEAFALAREAGHTVSIFLALETEEQALA...,0.0,0.848075,0.960055,1.477390,4.164045,89.762312,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,14.174671,-E--EEE----HHHHHHHHHHHHHHHHH--EEEEEEE---HHHHHH...,...,0.047259,1863.334870,0.246581,1041.793189,0.137864,-29710.339901,4784.881663,-25027.928821,9917.909186,-39628.249087
ovo_ogc_01_067_seq08,MVSTVIATPLSEEAKELLEEAFKLAEEGGFTVSLLLQLETEEQALK...,0.0,0.844367,1.058851,1.261072,4.429759,89.056200,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,14.174671,-E--EEE----HHHHHHHHHHHHHHHHH--EEEEEEE---HHHHHH...,...,0.007476,2252.990765,0.297020,1683.718345,0.221971,-44052.530867,2235.727977,-31370.428121,5266.649319,-49319.180186
ovo_ogc_01_071_seq02,EEALRAILEESARLLFERLKKMSKDASLEEKEKALEAAFEESVERM...,0.0,0.761071,1.861249,1.341439,4.908356,86.014998,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,15.269005,-HHHHHHHHHHHHHHHHHH--------HHHHHHHHHHHHHHHHHHH...,...,0.016168,1690.382936,0.202248,740.165923,0.088558,-34710.145256,2519.985988,-21934.183081,5209.900459,-39920.045715
ovo_ogc_01_071_seq05,AAALDAILREGARLLFEELRKMSPDASLEEKRAHLRAAVERMVEHI...,0.0,0.862591,0.489201,1.232468,3.259479,93.455410,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,15.269005,-HHHHHHHHHHHHHHHHHH--------HHHHHHHHHHHHHHHHHHH...,...,0.010219,1717.044833,0.209023,689.346088,0.083917,-38775.097609,1313.948940,-23158.278681,2896.636517,-41671.734126


In [12]:
values.to_csv('../../data/results/02_rfdiffusion_motif_scaffolding/descriptors.csv')

### Save descriptors for all designs including not accepted

In [13]:
all_values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS),
    descriptor_keys=[d.key for d in descriptors_proteinqc.SEQ_COMPOSITION_DESCRIPTORS]
)
print(len(all_values), 'designs')
print(len(all_values.columns), 'descriptors')
all_values.head()

6000 designs
43 descriptors


,Sequence A,Sequence length,Ala %,Cys %,Asp %,Glu %,Phe %,Gly %,His %,Ile %,...,MEC reduced,MEC cystines,Helix-forming residues %,Sheet-forming residues %,Turn-forming residues %,Flexibility average,GRAVY,Instability index,Molecular weight,Sequence entropy
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_jov_01_001_seq01,MEEFEAMAERYIAELRARGELKKAEEVEEVLKEMKEKGLTIKFFTV...,150,11.333333,0.0,4.000000,20.000000,2.000000,3.333333,0.666667,8.666667,...,13980,13980,54.666667,32.666667,14.666667,1.017276,-0.546000,47.235333,17069.2661,3.132263
ovo_jov_01_001_seq02,MEEFKKMFDEYIEKLRKEGKLEEAREVEEVRKLMEEKGLTIKRFRV...,150,13.333333,0.0,6.000000,16.666667,2.000000,3.333333,0.666667,6.000000,...,13980,13980,51.333333,32.666667,14.666667,1.009507,-0.382000,39.248000,17030.2942,3.192905
ovo_jov_01_002_seq01,MTLVTLTAEEIEELLGELSPRAKERTKSITVNLESGEVIVEEDGWL...,150,4.000000,0.0,5.333333,22.666667,2.666667,6.000000,2.000000,6.000000,...,9970,9970,46.000000,33.333333,18.000000,1.020292,-0.921333,60.564000,17661.5250,3.142495
ovo_jov_01_002_seq02,MTIVTLTAEQIEEMLGHLSEKAKERTKEIDIDTDSGQEIVERDGWL...,150,4.000000,0.0,7.333333,17.333333,3.333333,5.333333,3.333333,8.666667,...,12490,12490,41.333333,36.666667,17.333333,1.017534,-0.800667,45.386667,17526.5522,3.211456
ovo_jov_01_003_seq01,MITEKERWEIHKSKVEERDGIVYAEIELTNLDTGEKLKVKVKFEIE...,150,5.333333,0.0,3.333333,20.666667,1.333333,4.000000,1.333333,4.666667,...,9970,9970,50.666667,37.333333,12.666667,1.018203,-0.706667,53.905333,17370.6563,3.152267


In [14]:
all_values.to_csv('../../data/results/02_rfdiffusion_motif_scaffolding/seq_descriptors_all_designs.csv')